# Roll-for-Sandwich ratings — EDA

The ["Roll for Sandwich"](https://docs.google.com/spreadsheets/d/11fjXIUbUae35AFNlp-f10t3hSjbq5tiBykLA_IbUojs) project rolls
dice to pick a sandwich's bread, main, cheese, roughage, wild magic, and sauce, then rates the result 0-10.
This reframes the imbalance question as a **skewed regression** (plan.md 3.3): ratings pile up high (~50% above 7)
while low scores are rare (~11% below 3), so the "negative" examples are the low-rated sandwiches.

This notebook reproduces the reference EDA (rating distribution, ratings over time, per-ingredient rating spreads),
trains an XGBoost regressor on the multi-hot ingredient encoding, checks it on near-identical **matched pairs**, and
runs the **deli-counter** search for the best and worst sandwich the model can build. The plotting helpers live in
`negative_ml.viz.sandwiches`, so this notebook and `scripts/sandwiches_eda_model.py` draw the same figures.

In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from negative_ml.core import eval as ev
from negative_ml.core.config import (
    SANDWICH_FREQ_THRESHOLD_VIOLIN,
    SANDWICH_LOW_RATING_THRESHOLD,
    SANDWICH_RATING_RANGE,
    SANDWICH_XGB_PARAMS,
)
from negative_ml.core.splitting import random_split
from negative_ml.domains import sandwiches
from negative_ml.models import model_factory
from negative_ml.viz import sandwiches as viz

ds = sandwiches.load()
frame = sandwiches.clean_ratings(sandwiches.read_ratings())
n_low = int(ds.minority_mask(SANDWICH_LOW_RATING_THRESHOLD).sum())
print(
    f"{ds.n_samples} sandwiches; mean rating {ds.y.mean():.2f}; "
    f"{n_low} rated <= {SANDWICH_LOW_RATING_THRESHOLD:g} (the rare low-rated examples)"
)
frame.head()

## 1. Rating distribution

Ratings are skewed high with a thin low tail — exactly the imbalance the study turns on, now in a regression setting.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
viz.plot_rating_distribution(ds.y, ax)
plt.show()

## 2. Ratings over time

Rating versus episode number across the show's run (non-numeric episode markers are dropped).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
viz.plot_rating_timeline(ds.meta["Episode"], ds.y, ax)
plt.show()

## 3. Which ingredient values are well sampled?

The combinatorial menu is sparsely explored, so most ingredient values appear once or twice. The EDA and the deli
search restrict to values seen at least a handful of times (`SANDWICH_FREQ_THRESHOLD_VIOLIN`).

In [ ]:
frequent = sandwiches.ingredient_value_counts(frame, SANDWICH_FREQ_THRESHOLD_VIOLIN)
{col: len(values) for col, values in frequent.items()}

## 4. Rating by ingredient (violin + swarm)

How the rating distribution shifts with each ingredient choice, restricted to the well-sampled values. A violin shows
the spread; the jittered points are the individual sandwiches.

In [ ]:
for col in sandwiches.INGREDIENT_COLS:
    values = frequent[col]
    if not values:
        continue
    sub = frame[frame[col].isin(values)]
    fig, ax = plt.subplots(figsize=(max(4, 0.6 * len(values)), 4))
    viz.plot_ingredient_violin(
        sub[col].to_numpy(), sub["Clean_Rating"].to_numpy(), ax, col
    )
    plt.tight_layout()
    plt.show()

## 5. Multi-hot encoding + an XGBoost rating regressor

Each ingredient value becomes its own indicator column, so a sandwich is a multi-hot row with six ones. We fit the
house XGBoost regressor on a random split — with small, shallow `SANDWICH_XGB_PARAMS`, since a few hundred sandwiches
and hundreds of one-hot columns would overfit the baseball-scale defaults. This is a genuinely hard, sparse problem,
so the held-out correlation is modest.

In [ ]:
train_idx, test_idx = random_split(ds)
model = model_factory("regression")(**SANDWICH_XGB_PARAMS)
model.fit(ds.X[train_idx], ds.y[train_idx])

y_test = ds.y[test_idx]
y_pred = np.clip(model.predict(ds.X[test_idx]), *SANDWICH_RATING_RANGE)

fig, ax = plt.subplots(figsize=(5, 5))
viz.plot_predicted_vs_actual(y_test, y_pred, ax)
plt.show()

## 6. Matched pairs: near-identical sandwiches

Matched pairs are sandwiches that differ in only one or a few ingredients — near-twins whose ratings can still
diverge. Matched-pair accuracy asks whether a model fit on all the data ranks the higher-rated twin above its pair,
probing fine-grained discrimination rather than broad class separation. The menu is sparse, so the closest real pairs
differ in three of six ingredients.

In [ ]:
full = model_factory("regression")(**SANDWICH_XGB_PARAMS)
full.fit(ds.X, ds.y)

pairs = sandwiches.matched_pairs(ds)
acc = ev.matched_pair_accuracy(full.predict(ds.X), pairs, ds.y)
print(f"matched-pair accuracy = {acc:.3f} over {len(pairs)} pairs")
for i, j in pairs:
    a = " / ".join(ds.meta.iloc[i][sandwiches.INGREDIENT_COLS])
    b = " / ".join(ds.meta.iloc[j][sandwiches.INGREDIENT_COLS])
    print(f"  {ds.y[i]:.1f}: {a}")
    print(f"  {ds.y[j]:.1f}: {b}\n")

## 7. Deli counter: the best and worst sandwich

Enumerate every sandwich the model can build from the popular ingredients and ask it for the highest- and
lowest-scoring one (predictions clipped to the 0-10 scale). The reference notebook scored these one row at a time;
here the whole candidate menu is predicted in a single batch.

In [ ]:
value_lists = [frequent[col] for col in sandwiches.INGREDIENT_COLS]
candidates = pd.DataFrame(
    list(product(*value_lists)), columns=sandwiches.INGREDIENT_COLS
)
X_cand, _ = sandwiches.encode_ingredients(candidates, feature_names=ds.feature_names)
scores = np.clip(full.predict(X_cand), *SANDWICH_RATING_RANGE)

best, worst = int(scores.argmax()), int(scores.argmin())
best_label = " / ".join(candidates.iloc[best])
worst_label = " / ".join(candidates.iloc[worst])
print(f"searched {len(candidates):,} sandwiches")
print(f"best  {scores[best]:.2f}: {best_label}")
print(f"worst {scores[worst]:.2f}: {worst_label}")

fig, ax = plt.subplots(figsize=(7, 3))
viz.plot_best_worst_sandwich(
    best_label, float(scores[best]), worst_label, float(scores[worst]), ax
)
plt.tight_layout()
plt.show()